In [ ]:
# debug_syntax_deriver_db

import pandas as pd

In [ ]:
from pathlib import Path
import param
import panel as pn

pn.extension()
pn.config.sizing_mode="stretch_width"

block_size = 150  # what is the maximum context length for predictions?
learning_rate = 1e-4
n_embd = 1000
dropout = 0.2

class Settings(param.Parameterized):
    limit_count = param.Integer(1000 * 40, allow_None=True)

    n_embd = param.Integer(1000, label="n_embd")
    dropout = param.Number(0.2, label="dropout")
    n_head = param.Integer(10, label='n_head')
    block_size = param.Integer(150, label='block_size')  # the maximum context length for predictions
    dropout = param.Number(0.2, label='dropout')
    n_layer = param.Integer(10, label='n_layer')

    learning_rate = param.Number(1e-4,label='learning_rate')

    output_folder_path = param.Path(default=Path(".").resolve(), label='output_folder_path')
    mmx_file_path = param.Path(default=Path('set.new2023.mmx').resolve(), label='mmx_file_path')
    corpus01_file_path = param.Path(default=Path('corpus01.txt').resolve(), label='corpus01_file_path')
    corpus_folder_path = param.Path(default=Path('corpus').resolve(), label='corpus_folder_path')
    model_folder_path = param.Path(default=Path("model").resolve(), label='model_folder_path')

    def view(self):
        return pn.WidgetBox(
            pn.Column(pn.pane.Markdown("# Settings")),
            pn.Column(
                "## Parser03",
                self.param.limit_count,
            ),
            pn.Column(
                "## GPTLanguageModel",
                self.param.n_embd,
                self.param.n_head,
                self.param.block_size,
                self.param.dropout,
                self.param.n_layer,
            ),
            pn.Column(
                "## optimizer",
                self.param.learning_rate,
            ),
            pn.Column(
                "## paths",
                self.param.output_folder_path,
                self.param.mmx_file_path,
                self.param.corpus01_file_path,
                self.param.corpus_folder_path,
                self.param.model_folder_path,
            ),
            "## End",
            max_width=600,
        )

settings = Settings()
# settings.view()

In [ ]:
# load model
import torch

from source.shared import Encoder
from source.shared import load_model

from source.create_model import create_model

def set_up_model() -> Path:
    model_folder_path = Path(settings.model_folder_path)
    model_name = 'model.pt'
    model_file_path = model_folder_path.joinpath(model_name).resolve()
    if model_file_path.exists():
        print(f"model_file already exists: {model_file_path}")
    else:
        create_model(settings=settings)
    return model_file_path

model_file_path = set_up_model()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if torch.backends.mps.is_available():
    device = "mps"
print(f'device={device}')
encoder = Encoder.load_from_json(corpus_folder_path=settings.corpus_folder_path)
print(f'loading model and optimizer from checkpoint={model_file_path}')
model, optimizer = load_model(model_checkpoint_path=model_file_path, device=device, encoder=encoder)

# Create an example

In [71]:
from source.evaluate_model import ModelEvaluator

def make_syntax_deriver_db(max_examples: int):
    model_evaluator = ModelEvaluator(corpus_folder_path=settings.corpus_folder_path, model=model)
    model_evaluator.evaluate_model(max_examples=max_examples)
    syntax_deriver_db = model_evaluator.syntax_deriver.syntax_deriver_db
    return syntax_deriver_db

max_examples = 1
syntax_deriver_db = make_syntax_deriver_db(max_examples=max_examples)

vocab_size=160
epoch=520; step=5200; n_head=10; n_layer=10
assert_db_file_path=/Users/hale/PycharmProjects/MathAssertGPT/corpus/assert.db
=== start evaluate_model ===
max_val_examples=1


In [72]:
conn = syntax_deriver_db.conn
cursor = conn.cursor()
sql = 'SELECT id, statement, context, derivation, derivation_correct_count, syntax_deriver_error FROM math_statements ORDER BY id'
cursor.execute(sql)
math_statement_rows = cursor.fetchall()
example_count = len(math_statement_rows)
print(f'Example count: {example_count}')

Example count: 1


In [73]:
math_statement_row = math_statement_rows[0]
print(math_statement_row)

Row(id=1, statement="( Fun `' F = dom F -> ( `' A |` ran F ) )", context="|- \n|- ( Fun `' F = dom F -> ( `' A |` ran F ) ) <|over|>", derivation=None, derivation_correct_count=4, syntax_deriver_error='SyntaxDeriverWffRuleError')


In [74]:
derivation = math_statement_row.derivation
derivation_correct_count = math_statement_row.derivation_correct_count
statement_id = math_statement_row.id
context = math_statement_row.context
syntax_deriver_error = math_statement_row.syntax_deriver_error
wff_statement = math_statement_row.statement

print(f'statement_id: {statement_id}')
print(f'wff_statement: {wff_statement}')
print(f'derivation: {derivation}')
print(f'derivation_correct_count: {derivation_correct_count}')
print(f'syntax_deriver_error: {syntax_deriver_error}')
print(f'context:\n{context}')

statement_id: 1
wff_statement: ( Fun `' F = dom F -> ( `' A |` ran F ) )
derivation: None
derivation_correct_count: 4
syntax_deriver_error: SyntaxDeriverWffRuleError
context:
|- 
|- ( Fun `' F = dom F -> ( `' A |` ran F ) ) <|over|>


# Query

In [75]:
def beta(query):
    df = pd.read_sql_query(query, conn)
    return df

def phi(query, conn):
    df = pd.read_sql_query(query, conn)
    return pn.pane.DataFrame(df)

In [76]:
phi("SELECT * FROM math_statements", conn)

DataFrame(DataFrame, sizing_mode='stretch_width')

In [77]:
phi("SELECT * FROM rule_errors", conn)

DataFrame(DataFrame, sizing_mode='stretch_width')

In [78]:
def colorize(text, color):
    print(f'colorize text={text}')
    return f'<span style="color:{color}">{text}</span>'

dynamic_container = pn.Column(margin=(0, 0, 0, 0))

conn = syntax_deriver_db.conn
cursor = conn.cursor()
sql = 'SELECT id, statement, context, derivation, derivation_correct_count, syntax_deriver_error FROM math_statements ORDER BY id'
cursor.execute(sql)
math_statement_rows = cursor.fetchall()
for math_statement_index in range(len(math_statement_rows)):
    math_statement_row = math_statement_rows[math_statement_index]
    statement_id = math_statement_row.id
    statement = math_statement_row.statement
    context = math_statement_row.context
    derivation = math_statement_row.derivation
    derivation_correct_count = math_statement_row.derivation_correct_count
    syntax_deriver_error = math_statement_row.syntax_deriver_error
    prompt = context.split('\n')[0]
    lines = []
    lines.append(f'prompt: {prompt}')
    lines.append(f'predicted_statement: {statement}')
    lines.append(f'error: {syntax_deriver_error}')
    lines.append(f'derivation_correct_count={derivation_correct_count}')
    dynamic_container.append(pn.pane.Str('\n'.join(lines)))
    print(f'statement: {statement}')

    if derivation is None:
        lines = []
        lines.append(f'--- Possible continuations ---')
        dynamic_container.append(pn.pane.Str('\n'.join(lines)))
        sql = f'SELECT statement_id, rule_name, rule, mark_index, rule_tokens, current_rule_tokens FROM rule_errors WHERE statement_id = {statement_id} ORDER BY id'
        cursor.execute(sql)
        rule_error_rows = cursor.fetchall()
        for i in range(len(rule_error_rows)):
            lines = []
            rule_error_row = rule_error_rows[i]
            rule_tokens = rule_error_row.rule_tokens
            rule_name = rule_error_row.rule_name
            rule = rule_error_row.rule
            mark_index = rule_error_row.mark_index
            print(f'derivation_correct_count={derivation_correct_count}')
            print(f'mark_index: {mark_index}')
            token_index = derivation_correct_count
            print(f'token_index={token_index}')
            current_rule_tokens = rule_error_row.current_rule_tokens
            print(f'current_rule_tokens={current_rule_tokens}')
            current_token_index = max(0, derivation_correct_count + mark_index)
            print(f'current_token_index={current_token_index}')
            statement_split = statement.split()
            rule_split = rule.split()
            accumulated = " ".join(statement_split[:token_index]) + " "
            peeked = " ".join(statement_split[token_index: current_token_index]) + " "
            # current_token = statement[current_token_index]
            # current_token = f'{statement[current_token_index]} '
            current_token = " ".join(statement_split[current_token_index: current_token_index+1]) + " "
            rest = " ".join(statement_split[current_token_index+1:])
            colorize_text = f'{colorize(text=accumulated, color="blue")}{colorize(text=peeked, color="#00BB00")}{colorize(current_token, "red")}{colorize(text=rest, color="black")}'
            print(colorize_text)
            print(f'accumulated: {accumulated}')
            print(f'current_token: {current_token}')
            print(f'peeked: {peeked}')
            print(f'rest: {rest}')
            lines.append(f'Rule {rule_name}: {rule} mark_index={mark_index}')
            lines.append(f'Expected: {rule_split[mark_index]}')
            subpanel = pn.Column(
                # pn.pane.Str(f'===== Example {i + 1} error: ? ====='),
                pn.pane.Str('\n'.join(lines)),
                pn.pane.Markdown(
                    # Metamath color is Color(red: 0.933, green: 1, blue: 0.98, alpha: 1) red: EF green: FF blue: FB '#effffb'
                    colorize_text, styles={'font-family': 'monospace', 'font-size': '12pt', 'background-color': '#effffb'}
                )
            )
            dynamic_container.append(subpanel)

statement: ( Fun `' F = dom F -> ( `' A |` ran F ) )
derivation_correct_count=4
mark_index: 2
token_index=4
current_rule_tokens== dom F -> ( `' A |` ran F ) )
current_token_index=6
colorize text=( Fun `' F 
colorize text== dom 
colorize text=F 
colorize text=-> ( `' A |` ran F ) )
<span style="color:blue">( Fun `' F </span><span style="color:#00BB00">= dom </span><span style="color:red">F </span><span style="color:black">-> ( `' A |` ran F ) )</span>
accumulated: ( Fun `' F 
current_token: F 
peeked: = dom 
rest: -> ( `' A |` ran F ) )


In [79]:
outer_style = {
    # 'background': 'black',
    'border': '0px solid black',
    'padding': '5px',
    'margin': "0px",
}

pn.Column (
    dynamic_container,
    styles=outer_style,
)

Column(sizing_mode='stretch_width', styles={'border': '0px solid blac...})
    [0] Column(margin=(0, 0, 0, 0), sizing_mode='stretch_width')
        [0] Str(str, sizing_mode='stretch_width')
        [1] Str(str, sizing_mode='stretch_width')
        [2] Column(sizing_mode='stretch_width')
            [0] Str(str, sizing_mode='stretch_width')
            [1] Markdown(str, sizing_mode='stretch_width', styles={'font-family': 'monospace...})